# Analysing Model Output and Loss
to identify the src of mean static pose

In [ ]:
import os, glob, torch
from pathlib import Path
from torch.utils.data import DataLoader
from include.dataset import ViMoDataset
from include.model import ViMoFrameWorkv1, ViMoFrameWorkv2 
from include.ddpm_utils import make_beta_schedule, forward_diffusion_sample
from include.loss_utils import compute_losses, compact_optim_str
from adan_pytorch import Adan

torch.set_printoptions(precision=6, linewidth=1000, profile="default", sci_mode=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ViMoFrameWorkv2(motion_dim=151, pose_dim=17*3, max_T=1000, embed_dim=128,
                    	n_heads=4, m_nlayers=3, p_nlayers=2, cond_drop=0.25)
optimizer = Adan(model.parameters(), lr=1e-4, weight_decay=0.02)

DATA_DIR = Path.cwd() / "datasets"
processed_data_dir = DATA_DIR / "AIST/processed"
OUT_DIR = Path.cwd() / "outputs"
run_id = "_v2(sq-cap1.5-div)" # change here
RUN_DIR = MODEL_DIR = OUT_DIR / f"{model.compact_name}_{compact_optim_str(optimizer)}/run{run_id}"
MODEL_DIR = RUN_DIR / "models"
number = 5 # change here
model_path = MODEL_DIR / (f"Epoch{number}" + "_ckpt.pth")
ckpt = torch.load(model_path, map_location=device)
model.load_state_dict(ckpt['model_state'])

files = sorted(glob.glob(os.path.join(processed_data_dir, "*.npz")))
ds = ViMoDataset(files, t_views=9) # all files loader
dl = DataLoader(ds, batch_size=1, shuffle=True, pin_memory=True)
it = iter(dl)

In [36]:
batch = next(it)
T = model.max_T
make_beta_schedule(T, device=device)

model.to(device) # move to target device
m_gt = batch['m3d'].to(device) # (B, S=150, 151 => 24*4 + 4 + 3)
p2d = batch['p2d'].to(device) # (B, S=150, 51 => 17*3)

t = torch.randint(0, T, (dl.batch_size,), device=device)
xt = forward_diffusion_sample(m_gt, t)
m_pred = model(xt, t, p2d)

In [37]:
diff = (m_gt - m_pred).squeeze() # remove batch dim for size=1 => shape(150,151)

# count masks
pos_mask = diff > 0
neg_mask = diff < 0

# totals
total_pos = int(pos_mask.sum().item())
total_neg = int(neg_mask.sum().item())

#  per-sample counts across the last dim -> shape (S)
pos_counts_per_frame = pos_mask.sum(dim=-1)
neg_counts_per_frame = neg_mask.sum(dim=-1)

print(f"diff.shape = {diff.shape}")
print(f"Total positives: {total_pos}, Total negatives: {total_neg}")
print(f"Pos counts per Frame (S) mean = {pos_counts_per_frame.float().mean():.4f}, Neg counts per Frame (S) mean = {neg_counts_per_frame.float().mean():.4f}")
print(f"diff: \n{diff}")

diff.shape = torch.Size([150, 151])
Total positives: 11714, Total negatives: 10936
Pos counts per Frame (S) mean = 78.0933, Neg counts per Frame (S) mean = 72.9067
diff: 
tensor([[-0.518304, -0.079146,  1.102632,  ...,  0.079567, -0.028874,  0.604923],
        [-0.597811, -0.088891,  1.062063,  ...,  0.131445, -0.048586,  0.610938],
        [-0.567987,  0.013847,  0.952670,  ...,  0.057196, -0.077502,  0.612374],
        ...,
        [ 0.457228,  0.230908, -0.170261,  ...,  0.103685, -0.074641, -0.161120],
        [ 0.426970,  0.184866, -0.092455,  ...,  0.124894, -0.073569, -0.160141],
        [ 0.466879,  0.138370, -0.095930,  ...,  0.134229, -0.043549, -0.120766]], device='cuda:0', grad_fn=<SqueezeBackward0>)


In [38]:
sqr = torch.square(diff)
print(sqr)

tensor([[0.268639, 0.006264, 1.215797,  ..., 0.006331, 0.000834, 0.365932],
        [0.357378, 0.007902, 1.127978,  ..., 0.017278, 0.002361, 0.373245],
        [0.322609, 0.000192, 0.907579,  ..., 0.003271, 0.006007, 0.375002],
        ...,
        [0.209057, 0.053318, 0.028989,  ..., 0.010751, 0.005571, 0.025960],
        [0.182303, 0.034176, 0.008548,  ..., 0.015599, 0.005412, 0.025645],
        [0.217976, 0.019146, 0.009203,  ..., 0.018017, 0.001897, 0.014585]], device='cuda:0', grad_fn=<PowBackward0>)


In [13]:
print("mean =", sqr.mean())

mean = tensor(0.6573, device='cuda:0', grad_fn=<MeanBackward0>)
